# BP2 — Keyword Mention Analysis

**Purpose**: Analyze how debt sustainability topics have evolved in IMF documents (DSAs, AIVs, Programs) for 68 PRGT-eligible countries, 2005–2023.

This notebook uses shared modules from `E:\Data\gsoto\analytical\`.

In [1]:
# =============================================================================
# CELL 1: Import shared modules
# =============================================================================
import sys
sys.path.insert(0, r'E:\Data\gsoto\analytical')

from config import *
from topics import MENTION_TOPICS, COUNTRY_TOPIC_GROUPS
from data import build_all
from search import (build_mentions_dsa_md, build_mentions_aiv,
                    build_mentions_programs, combine_mentions,
                    diagnose_keyword_hits)

print(f"Topics loaded: {len(MENTION_TOPICS)}")
print(f"Topic groups: {len(COUNTRY_TOPIC_GROUPS)}")

Topics loaded: 68
Topic groups: 18


In [2]:
# =============================================================================
# CELL 2: BP2-specific output directories
# =============================================================================
OUTPUT_DIR = Path("U:/dil/bp2/output")
OUTPUT_DIR_CHARTS = Path("U:/dil/bp2/output/charts")
#DSA_TXT_DIR = Path("U:/dil/bp2/output/extracted_texts_clean")

for d in [OUTPUT_DIR, OUTPUT_DIR_CHARTS]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Output:  {OUTPUT_DIR}")
print(f"Charts:  {OUTPUT_DIR_CHARTS}")
#print(f"DSA TXT: {DSA_TXT_DIR}")

Output:  U:\dil\bp2\output
Charts:  U:\dil\bp2\output\charts


## 1. Build Core Dataframes

Loads country characteristics, scans document folders, builds inventory and time series panels.

In [3]:
# =============================================================================
# CELL 3: Build all dataframes
# =============================================================================
result = build_all(
    COUNTRY_CHAR_PATH, ARRANGEMENT_NUMBERS,
    INPUT_DIR_DSA_PDF, INPUT_DIR_AIV, INPUT_DIR_PROGRAMS
)

# Unpack
df_base       = result['df_base']
df_inventory  = result['df_inventory']
df_ts         = result['df_ts']
df_dsa        = result['df_dsa']
df_aiv        = result['df_aiv']
df_programs   = result['df_programs']
dsa_registry  = result['dsa_registry']
aiv_registry  = result['aiv_registry']
prog_registry = result['prog_registry']
year_to_arr   = result['year_to_arr']

print(f"\ndf_base:      {df_base.shape}")
print(f"df_inventory: {df_inventory.shape}")
print(f"df_ts:        {df_ts.shape}")
print(f"df_dsa:       {df_dsa.shape}")
print(f"df_aiv:       {df_aiv.shape}")
print(f"df_programs:  {df_programs.shape}")

E:\Data\gsoto\analytical\data.py:92: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_base[col] = df_base[col].replace('na', np.nan).fillna(0).astype(int)


⚠ Programs skipped (no metadata date): 512
DSA: 924 docs | AIV: 733 docs | Programs: 944 docs
DOCUMENT SUMMARY
  DSA: 924 docs | Matched: 924 | Years: 2005-2024
  AIV: 733 docs | Matched: 733 | Years: 2005-2025
  Programs: 944 docs | Matched: 944 | Years: 2005-2025

df_base:      (69, 18)
df_inventory: (69, 21)
df_ts:        (1449, 23)
df_dsa:       (924, 18)
df_aiv:       (733, 15)
df_programs:  (944, 18)


In [ ]:
# =============================================================================
# CELL 4: Export inventory (optional)
# =============================================================================
with pd.ExcelWriter(OUTPUT_DIR / 'document_inventory.xlsx') as writer:
    df_inventory.to_excel(writer, sheet_name='By Country', index=False)
    df_ts.to_excel(writer, sheet_name='By Country-Year', index=False)
print(f"✓ Saved: {OUTPUT_DIR / 'document_inventory.xlsx'}")

## 2. Keyword Search

Run keyword matching across all document types: DSA (TXT), AIV (MD), Programs (MD).

In [ ]:
# =============================================================================
# CELL 5: Run keyword search on all 3 doc types
# =============================================================================

# 1. DSA (from MDs — Docling, cleaner than TXT)
df_mentions_dsa = build_mentions_dsa_md(INPUT_DIR_DSA_MD, df_dsa, MENTION_TOPICS)

# 2. AIV (from Markdowns)
df_mentions_aiv = build_mentions_aiv(df_aiv, MENTION_TOPICS)

# 3. Programs (from Markdowns)
df_mentions_prog = build_mentions_programs(df_programs, MENTION_TOPICS)

# 4. Combine + merge country characteristics
df_mentions_all = combine_mentions(
    df_mentions_dsa, df_mentions_aiv, df_mentions_prog, df_inventory
)


# 5. Save
#df_mentions_all.to_csv(OUTPUT_DIR / 'mentions_all_doctypes.csv', index=False)
#print(f"\n✓ Saved: {OUTPUT_DIR / 'mentions_all_doctypes.csv'}")

In [13]:
# 1. Verificar que existe y se ve razonable
print(df_mentions_all['word_count'].describe())

# 2. Por doc_type
print(df_mentions_all.groupby('doc_type')['word_count'].describe().round(0))

# 3. Documentos sospechosamente cortos
short = df_mentions_all[df_mentions_all['word_count'] < 500]
if len(short) > 0:
    print(f"\n⚠ {len(short)} docs con menos de 500 palabras:")
    print(short[['doc_type', 'country', 'year', 'filename', 'word_count']])

count     2601.000000
mean     14735.615917
std      11177.758940
min        605.000000
25%       4207.000000
50%      13691.000000
75%      21730.000000
max      77171.000000
Name: word_count, dtype: float64
          count     mean     std     min      25%      50%      75%      max
doc_type                                                                    
AIV       733.0  18570.0  9773.0  4298.0  11756.0  15949.0  22254.0  75256.0
DSA       924.0   3580.0  1678.0   605.0   2577.0   3366.0   4445.0  23700.0
Program   944.0  22677.0  8432.0  6190.0  16602.0  21302.0  27724.0  77171.0


## 3. Inventory & Stylized Facts

Descriptive charts of the document corpus.

In [ ]:
df_plot = df_ts[df_ts['year'] <= 2024]
by_year = df_plot.groupby('year')[['n_dsa', 'n_aiv', 'n_programs']].sum()
total = by_year.sum(axis=1)

fig, ax = plt.subplots(figsize=(14, 5))
b1 = ax.bar(by_year.index, by_year['n_dsa'], label='DSA', color='#4472C4')
b2 = ax.bar(by_year.index, by_year['n_aiv'], bottom=by_year['n_dsa'], label='AIV', color='#ED7D31')
b3 = ax.bar(by_year.index, by_year['n_programs'], bottom=by_year['n_dsa'] + by_year['n_aiv'], label='Program Reports', color='#70AD47')

# Números dentro de cada segmento
for bars, col in [(b1, 'n_dsa'), (b2, 'n_aiv'), (b3, 'n_programs')]:
    for bar, val in zip(bars, by_year[col]):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_y() + bar.get_height()/2,
                    str(val), ha='center', va='center', fontsize=10, color='white')

# Total encima en negrita
for x, t in zip(by_year.index, total):
    ax.text(x, t + 2, str(t), ha='center', fontsize=8, fontweight='bold')

ax.axhline(total.mean(), color='gray', linestyle='--', alpha=0.5, label=f'Avg ({total.mean():.0f})')
ax.set_title('Total Documents per Year (All Countries, 2005-2024)')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Documents')
ax.set_xticks(range(2005, 2025))
ax.legend()
plt.tight_layout()
fig.text(0.01, -0.02, 'Source: IEO Staff\'s calculations based on Debt Sustainability Analysis, Program and Article IV Consultation public documents.', 
         fontsize=10, fontstyle='italic', color='black')
#plt.savefig(OUTPUT_DIR_CHARTS / 'total_documents_per_year.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df_plot = df_ts[df_ts['year'] <= 2024]
years = range(2005, 2025)
width = 0.35

fig, ax = plt.subplots(figsize=(18, 6))

for i, (val, label, offset) in enumerate([(1, 'HIPC', -width/2), (0, 'Non-HIPC', width/2)]):
    group = df_plot[df_plot['hipc'] == val]
    by_year = group.groupby('year')[['n_dsa', 'n_aiv', 'n_programs']].sum().reindex(years, fill_value=0)
    x = np.array(list(years)) + offset

    b1 = ax.bar(x, by_year['n_dsa'], width, label=f'{label} – DSA', color='#4472C4', alpha=1 if val == 1 else 0.5)
    b2 = ax.bar(x, by_year['n_aiv'], width, bottom=by_year['n_dsa'], label=f'{label} – AIV', color='#ED7D31', alpha=1 if val == 1 else 0.5)
    b3 = ax.bar(x, by_year['n_programs'], width, bottom=by_year['n_dsa'] + by_year['n_aiv'], label=f'{label} – Programs', color='#70AD47', alpha=1 if val == 1 else 0.5)

    total = by_year.sum(axis=1)
    for xi, t in zip(x, total):
        if t > 0:
            ax.text(xi, t + 1, str(t), ha='center', fontsize=7, fontweight='bold')

n_hipc = df_plot[df_plot['hipc'] == 1]['ifs'].nunique()
n_non = df_plot[df_plot['hipc'] == 0]['ifs'].nunique()

ax.set_title(f'Total Documents per Year: HIPC (n={n_hipc}) vs Non-HIPC (n={n_non})', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Documents')
ax.set_xticks(years)
ax.tick_params(axis='x', rotation=45)
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
fig.text(0.01, -0.02, "Source: IEO Staff's calculations based on DSA, Program and AIV public documents.",
         fontsize=10, fontstyle='italic', color='black')
#plt.savefig(OUTPUT_DIR_CHARTS / 'documents_hipc_grouped.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df_plot = df_ts[df_ts['year'] <= 2024]
years = sorted(df_plot['year'].unique())

fig, ax = plt.subplots(figsize=(14, 5))

for val, label, color in [(1, 'HIPC', '#4472C4'), (0, 'Non-HIPC', '#ED7D31')]:
    group = df_plot[df_plot['hipc'] == val]
    n_countries = group['ifs'].nunique()

    totals = []
    for y in years:
        g = group[group['year'] == y]
        n = g['ifs'].nunique()
        t = g['n_dsa'].sum() + g['n_aiv'].sum() + g['n_programs'].sum()
        totals.append(t / n if n > 0 else 0)

    totals = np.array(totals)
    smooth = pd.Series(totals).rolling(window=3, center=True).mean().values

    ax.scatter(years, totals, color=color, s=35, alpha=0.5, zorder=3)
    ax.plot(years, smooth, '-', color=color, linewidth=2.5, label=f'{label} (n={n_countries})', zorder=2)

ax.set_title('Average Total Documents per Country: HIPC vs Non-HIPC',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Avg Documents per Country\n(DSA + AIV + Programs)', fontsize=11)
ax.set_xticks(years)
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.legend(fontsize=11)

fig.text(0.01, -0.08,
         'Note: Smoothing via 3-year centered moving average. Points show original annual values.\n'
         'Source: IEO Staff\'s calculations from IMF DSA, AIV, and Program documents.',
         fontsize=8, ha='left')

plt.tight_layout()
plt.subplots_adjust(bottom=0.15)
#plt.savefig(OUTPUT_DIR_CHARTS / 'documents_avg_hipc_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from scipy import stats

df_plot_grp = df_ts[df_ts['year'] <= 2024]
years = sorted(df_plot_grp['year'].unique())

groupings = [
    ('cex', 'Commodity Exporter', 'CEX', 'Non-CEX'),
    ('fcs', 'Fragile/Conflict State', 'FCS', 'Non-FCS'),
    ('sds', 'Small Developing State', 'SDS', 'Non-SDS'),
    ('rst', 'RST', 'RST', 'Non-RST'),
    ('frontier_market', 'Frontier Market', 'Frontier', 'Non-Frontier'),
    ('exposure_china', 'China Exposure', 'China Exp.', 'Non-China Exp.'),
    ('hipc', 'HIPC', 'HIPC', 'Non-HIPC'),
]

for col, title, yes_label, no_label in groupings:
    fig, ax = plt.subplots(figsize=(14, 5))

    diffs = []
    for y in years:
        g_yes = df_plot_grp[(df_plot_grp['year'] == y) & (df_plot_grp[col] == 1)]
        g_no  = df_plot_grp[(df_plot_grp['year'] == y) & (df_plot_grp[col] == 0)]
        n_yes = g_yes['ifs'].nunique()
        n_no  = g_no['ifs'].nunique()
        avg_yes = g_yes['n_dsa'].sum() / n_yes if n_yes > 0 else 0
        avg_no  = g_no['n_dsa'].sum() / n_no if n_no > 0 else 0
        diffs.append(avg_yes - avg_no)

    diffs = np.array(diffs)
    smooth = pd.Series(diffs).rolling(window=3, center=True).mean().values

    # Dotted line connecting original points
    ax.plot(years, diffs, '--', color='gray', linewidth=1, alpha=0.3, zorder=1)

    # Green/red points
    colors_pts = ['#2CA02C' if d >= 0 else '#D62728' for d in diffs]
    ax.scatter(years, diffs, c=colors_pts, s=50, zorder=3, edgecolors='white', linewidth=0.5)

    # Smoothed line
    ax.plot(years, smooth, '-', color='#4472C4', linewidth=2.5, zorder=2)
    # Zero line
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    # Shaded areas
    ax.fill_between(years, 0, smooth,
                    where=[s >= 0 if not np.isnan(s) else False for s in smooth],
                    color='#2CA02C', alpha=0.10, interpolate=True)
    ax.fill_between(years, 0, smooth,
                    where=[s < 0 if not np.isnan(s) else False for s in smooth],
                    color='#D62728', alpha=0.10, interpolate=True)

    # Annotate original values
    for y, d, c in zip(years, diffs, colors_pts):
        offset = 0.03 if d >= 0 else -0.03
        va = 'bottom' if d >= 0 else 'top'
        ax.text(y, d + offset, f'{d:.2f}', ha='center', va=va,
                fontsize=7, color=c, fontweight='bold')

    n_yes = df_plot_grp[df_plot_grp[col] == 1]['ifs'].nunique()
    n_no  = df_plot_grp[df_plot_grp[col] == 0]['ifs'].nunique()

    ax.set_title(f'DSA Coverage Gap: {title}\n'
                 f'Green = {yes_label} higher, Red = {no_label} higher',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Year', fontsize=11)
    ax.set_ylabel('Difference in Avg DSAs per Country\n(Yes − No)', fontsize=11)
    ax.set_xticks(years)
    ax.tick_params(axis='x', rotation=45, labelsize=9)

    fig.text(0.01, -0.10,
             f'Note: {yes_label} (n={n_yes}), {no_label} (n={n_no}). '
             'Categories are non-mutually exclusive.\n'
             'Smoothing: 3-year centered moving average. '
             'Points show original annual values.\n'
             'Source: IEO Staff\'s calculations from IMF DSA documents.',
             fontsize=8, ha='left')

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.20)
    #plt.savefig(OUTPUT_DIR_CHARTS / f'dsa_gap_{col}.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# DSAs por año por región — normalizado (2 paneles)
import seaborn as sns

df_heat_src = df_ts[(df_ts['year'] <= 2024) & (df_ts['region'] != 'EUR')]
years = sorted(df_heat_src['year'].unique())
regions = sorted(df_heat_src['region'].unique())

fig, ax = plt.subplots(figsize=(16, 5))

matrix = []
for region in regions:
    row = []
    for y in years:
        group = df_heat_src[(df_heat_src['year'] == y) & (df_heat_src['region'] == region)]
        n = group['ifs'].nunique()
        total = group['n_dsa'].sum() + group['n_aiv'].sum() + group['n_programs'].sum()
        row.append(total / n if n > 0 else 0)
    matrix.append(row)

df_heat = pd.DataFrame(matrix, index=regions, columns=years)

sns.heatmap(df_heat, ax=ax, cmap='YlOrRd', annot=True, fmt='.2f',
            linewidths=0.5, linecolor='white', cbar_kws={'shrink': 0.8},
            annot_kws={'fontsize': 10, 'fontweight': 'bold'})        # <-- tamaño y negrita de valores

ax.set_title('Average Documents (DSA + AIV + Program Reports) per Country by Region',
             fontsize=14, fontweight='bold')                          # <-- tamaño título
ax.set_xlabel('Year', fontsize=12)                                    # <-- tamaño eje X
ax.set_ylabel('', fontsize=12)                                        # <-- tamaño eje Y
ax.tick_params(axis='x', rotation=45, labelsize=10)                   # <-- tamaño etiquetas X
ax.tick_params(axis='y', labelsize=11)                                # <-- tamaño etiquetas Y

# Nota: EUR excluido
fig.text(0.01, -0.06,
         'Note: EUR region excluded (only 1 country: Moldova).',
         fontsize=9, fontstyle='italic', ha='left')                   # <-- tamaño nota

# Source / calculations
fig.text(0.01, -0.12,
         'Source: IMF DSA, AIV, and Program documents. '
         'Calculation: total documents per region-year divided by number of countries in region.',
         fontsize=9, ha='left')                                       # <-- tamaño source

plt.tight_layout()
plt.subplots_adjust(bottom=0.18)  # espacio para las anotaciones
#plt.savefig(OUTPUT_DIR_CHARTS / 'avg_document_count_region.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df_plot_reg = df_ts[(df_ts['year'] <= 2024) & (df_ts['region'] != 'EUR')]

fig, ax = plt.subplots(figsize=(14, 5))

regions = sorted(df_plot_reg['region'].unique())
periods = [('2005-09', 2005, 2009), ('2010-14', 2010, 2014),
           ('2015-19', 2015, 2019), ('2020-24', 2020, 2024)]
colors = ['#4472C4', '#ED7D31', '#70AD47', '#FFC000']

n_periods = len(periods)
width = 0.8 / n_periods
x = np.arange(len(regions))

for i, (label, y_start, y_end) in enumerate(periods):
    avgs = []
    for region in regions:
        group = df_plot_reg[(df_plot_reg['year'] >= y_start) & (df_plot_reg['year'] <= y_end)
                            & (df_plot_reg['region'] == region)]
        n = group['ifs'].nunique()
        total_docs = group['n_dsa'].sum() + group['n_aiv'].sum() + group['n_programs'].sum()
        n_years = y_end - y_start + 1
        avgs.append(total_docs / n / n_years if n > 0 else 0)
    bars = ax.bar(x + i * width - (n_periods - 1) * width / 2, avgs,
                  width=width, label=label, color=colors[i])
    # Valores sobre cada barra
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + 0.03,
                f'{height:.1f}', ha='center', va='bottom', fontsize=7)

ax.set_title('Average Documents (DSA + AIV + Program Reports) per Country per Year by Region', fontsize=13)
ax.set_xlabel('Region')
ax.set_ylabel('Avg Docs per Country per Year')
ax.set_xticks(x)
ax.set_xticklabels(regions, fontsize=10)
ax.legend(fontsize=10)
plt.tight_layout()
#plt.savefig(OUTPUT_DIR_CHARTS / 'avg_document_count_region_bars.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:

df_heat_src = df_ts[(df_ts['year'] <= 2024)]
years = sorted(df_heat_src['year'].unique())

groupings = [
    ('cex', 'Commodity Exporter'),
    ('fcs', 'Fragile/Conflict State'),
    ('sds', 'Small Developing State'),
    ('rst', 'RST'),
    ('frontier_market', 'Frontier Market'),
    ('exposure_china', 'China Exposure'),
    ('hipc', 'HIPC'),
]

for col, label in groupings:
    fig, ax = plt.subplots(figsize=(16, 3))

    matrix = []
    row_labels = []
    for val, tag in [(1, 'Yes'), (0, 'No')]:
        sub = df_heat_src[df_heat_src[col] == val]
        n_countries = sub['ifs'].nunique()
        row = []
        for y in years:
            group = sub[sub['year'] == y]
            n = group['ifs'].nunique()
            total = group['n_dsa'].sum() + group['n_aiv'].sum() + group['n_programs'].sum()
            row.append(total / n if n > 0 else 0)
        matrix.append(row)
        row_labels.append(f'{tag} (n={n_countries})')

    df_h = pd.DataFrame(matrix, index=row_labels, columns=years)

    sns.heatmap(df_h, ax=ax, cmap='YlOrRd', annot=True, fmt='.2f',
                linewidths=0.5, linecolor='white', cbar_kws={'shrink': 0.8},
                annot_kws={'fontsize': 10, 'fontweight': 'bold'})

    ax.set_title(f'{label}: Avg Docs (DSA + AIV + Program Reports) per Country',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Year', fontsize=11)
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    ax.tick_params(axis='y', labelsize=10)

    fig.text(0.01, -0.15,
             'Note: Categories are non-mutually exclusive (a country can belong to multiple groups).\n'
         'Source: IEO Staff\'s calculations from the IMF Debt Sustainability Analysis, Article IV, and Program documents.\n'
         'Calculation: total documents / number of countries in group.',
             fontsize=8, ha='left')

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.25)
    plt.savefig(OUTPUT_DIR_CHARTS / f'heatmap_avg_{col}.png', dpi=300, bbox_inches='tight')
    plt.show()

### Page Counts

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.scatter(df_docs['year'], df_docs['n_pages'], 
           alpha=0.15, s=20, color='#4472C4', edgecolors='none')

# Smoothed median line
median_by_year = df_docs.groupby('year')['n_pages'].median()
smooth_median = median_by_year.rolling(window=3, center=True).mean()

ax.plot(smooth_median.index, smooth_median.values, '-', color='#D62728', 
        linewidth=2.5, label='Median (3yr MA)')

# Smoothed mean line
mean_by_year = df_docs.groupby('year')['n_pages'].mean()
smooth_mean = mean_by_year.rolling(window=3, center=True).mean()

ax.plot(smooth_mean.index, smooth_mean.values, '--', color='black', 
        linewidth=2, label='Mean (3yr MA)')

ax.set_title('DSA Document Length Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Number of Pages', fontsize=11)
ax.set_xticks(range(2005, 2025))
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.legend(fontsize=10)

fig.text(0.01, -0.08,
         f'Note: Each point is one DSA document (n={len(df_docs)}). '
         'Smoothing via 3-year centered moving average.\n'
         'Source: IEO Staff\'s calculations from IMF DSA documents.',
         fontsize=8, ha='left')

plt.tight_layout()
plt.subplots_adjust(bottom=0.15)
plt.show()

## 4. Visualization Functions

All chart functions for keyword mention analysis.

### 4b. IQR Comparison (Median + IQR / Mean)

In [13]:
#DONT RUN, DELETE, NOT NORMALIZED
def plot_topic_iqr_comparison(df_mentions_all, topic_key, topic_label, max_year=2023, save_dir=None, doc_type='DSA'):
    """Side by side: median with IQR (left) vs mean without band (right). Independent y-axes."""
    
    df = df_mentions_all[(df_mentions_all['doc_type'] == doc_type)
                          & (df_mentions_all['year'] <= max_year)].copy()
    col = f'count_{topic_key}'
    
    cy = df.groupby(['country', 'year']).agg(
        avg_hits=(col, 'mean')
    ).reset_index()
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))  # no sharey
    
    for ax, stat in zip(axes, ['median', 'mean']):
        yearly = cy.groupby('year')['avg_hits'].agg(
            center=stat,
            p25=lambda x: x.quantile(0.25),
            p75=lambda x: x.quantile(0.75),
        ).reset_index()
        
        if stat == 'median':
            ax.fill_between(yearly['year'], yearly['p25'], yearly['p75'],
                             alpha=0.2, color='tab:blue')
        
        ax.plot(yearly['year'], yearly['center'], '-o', linewidth=2.5,
                color='tab:blue', markersize=5, zorder=3)
        
        ax.set_title(f'{stat.capitalize()}', fontweight='bold', fontsize=13)
        ax.set_ylabel(f'{stat.capitalize()} hits per doc', fontsize=11)
        ax.set_ylim(bottom=0)
        ax.grid(axis='y', alpha=0.3)
        ax.set_xticks(range(2005, max_year + 1, 2))
        ax.set_xlim(2004.5, max_year + 0.5)
        ax.tick_params(axis='x', rotation=45, labelsize=10)
        ax.axvline(x=2017.5, color='red', linestyle=':', alpha=0.4, linewidth=1)
    
    fig.suptitle(f'{topic_label} — All LICs ({doc_type} only)\n'
             f'Keyword hits per doc (country-year level)',
             fontweight='bold', fontsize=14, y=1.02)
    
    from textwrap import fill
    note_text = (
        'Note: Each country contributes one observation per year, computed as the average keyword hits '
        'per document within that country-year. Left panel shows the median across countries with IQR band '
        '(P25–P75); right panel shows the mean. The median captures the typical country experience, '
        'while the mean reflects overall intensity including outliers. '
        'Red dotted line marks the 2018 LIC-DSF revision. '
        'DSA = Debt Sustainability Analysis; LIC-DSF = Low-Income Country Debt Sustainability Framework; '
        'IQR = Interquartile Range. '
        'Source: IEO staff calculations based on IMF DSA documents.'
    )
    fig.text(0.05, -0.06, fill(note_text, width=150), fontsize=9, ha='left', va='top')
    
    plt.tight_layout()
    
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fname = os.path.join(save_dir, f'iqr_comparison_{topic_key}_{doc_type}.png')
        fig.savefig(fname, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    else:
        plt.show()


# Run
#SAVE_DIR_IQR_V3 = str(OUTPUT_DIR / 'iqr_comparison_v3')

#for bucket_name, topics in COUNTRY_TOPIC_GROUPS.items():
#    for tk, tl in topics:
#        plot_topic_iqr_comparison(df_mentions_all, tk, tl, max_year=2023, save_dir=SAVE_DIR_IQR_V3)

#print(f"✓ Saved to {SAVE_DIR_IQR_V3}")

In [ ]:
for bucket_name, topics in COUNTRY_TOPIC_GROUPS.items():
    for tk, tl in topics:
        plot_topic_iqr_comparison(df_mentions_all, tk, tl, max_year=2023)

### 4c. IQR by Country Group

In [15]:
def plot_topic_iqr_by_group(df_mentions_all, topic_key, topic_label, 
                             group_col, group_label=None, max_year=2023, save_dir=None):
    """
    Side by side median vs mean, with two lines per panel:
    solid = characteristic is 1, dashed = characteristic is 0.
    """
    if group_label is None:
        group_label = group_col.upper()
    
    df = df_mentions_all[(df_mentions_all['doc_type'] == 'DSA')
                          & (df_mentions_all['year'] <= max_year)
                          & (df_mentions_all[group_col].notna())].copy()
    col = f'count_{topic_key}'
    
    # Step 1: country-year avg hits per doc
    cy = df.groupby(['country', 'year', group_col]).agg(
        avg_hits=(col, 'mean')
    ).reset_index()
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
    
    for ax, stat in zip(axes, ['median', 'mean']):
        for grp_val, style, color, label in [
            (1, '-o',  'tab:blue', f'{group_label} = Yes'),
            (0, '--s', 'tab:orange', f'{group_label} = No'),
        ]:
            sub = cy[cy[group_col] == grp_val]
            if len(sub) == 0:
                continue
            
            yearly = sub.groupby('year')['avg_hits'].agg(
                center=stat,
                p25=lambda x: x.quantile(0.25),
                p75=lambda x: x.quantile(0.75),
            ).reset_index()
            
            ax.fill_between(yearly['year'], yearly['p25'], yearly['p75'],
                             alpha=0.12, color=color)
            ax.plot(yearly['year'], yearly['center'], style, linewidth=2.5,
                    color=color, markersize=5, zorder=3, label=label)
        
        ax.set_title(f'{stat.capitalize()}', fontweight='bold', fontsize=13)
        ax.set_ylabel(f'{stat.capitalize()} hits per doc' if ax == axes[0] else '', fontsize=11)
        ax.set_ylim(bottom=0)
        ax.grid(axis='y', alpha=0.3)
        ax.set_xticks(range(2005, max_year + 1, 2))
        ax.set_xlim(2004.5, max_year + 0.5)
        ax.tick_params(axis='x', rotation=45, labelsize=10)
        ax.axvline(x=2017.5, color='red', linestyle=':', alpha=0.4, linewidth=1)
        ax.legend(fontsize=11, loc='best')
    
    fig.suptitle(f'{topic_label} — by {group_label}\nHits per doc (country-year) with IQR',
                 fontweight='bold', fontsize=14, y=1.02)
    
    fig.text(0.05, -0.06,
             'Note: Each country contributes one value per year (avg hits per doc). '
             'Shaded band = IQR (P25–P75). Red dotted line = 2018 LIC-DSF revision.\n'
             'Source: IEO Staff\'s calculations from IMF DSA documents.',
             fontsize=9, ha='left')
    
    plt.tight_layout()
    
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        safe_group = group_col.lower().replace(' ', '_')
        fname = os.path.join(save_dir, f'iqr_{safe_group}_{topic_key}.png')
        #fig.savefig(fname, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    else:
        plt.show()

### 4d. IQR by Group — All Doc Types

In [ ]:
def plot_topic_iqr_by_group_alldocs(df_mentions_all, topic_key, topic_label, 
                                     group_col, group_label=None, max_year=2023, save_dir=None):
    """
    Side by side median vs mean, with two lines per panel.
    Uses ALL doc types, not just DSA.
    """
    if group_label is None:
        group_label = group_col.upper()
    
    df = df_mentions_all[(df_mentions_all['year'] <= max_year)
                          & (df_mentions_all[group_col].notna())].copy()
    col = f'count_{topic_key}'
    
    cy = df.groupby(['country', 'year', group_col]).agg(
        avg_hits=(col, 'mean')
    ).reset_index()
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
    
    for ax, stat in zip(axes, ['median', 'mean']):
        for grp_val, style, color, label in [
            (1, '-o',  'tab:blue', f'{group_label} = Yes'),
            (0, '--s', 'tab:orange', f'{group_label} = No'),
        ]:
            sub = cy[cy[group_col] == grp_val]
            if len(sub) == 0:
                continue
            
            yearly = sub.groupby('year')['avg_hits'].agg(
                center=stat,
                p25=lambda x: x.quantile(0.25),
                p75=lambda x: x.quantile(0.75),
            ).reset_index()
            
            ax.fill_between(yearly['year'], yearly['p25'], yearly['p75'],
                             alpha=0.12, color=color)
            ax.plot(yearly['year'], yearly['center'], style, linewidth=2.5,
                    color=color, markersize=5, zorder=3, label=label)
        
        ax.set_title(f'{stat.capitalize()}', fontweight='bold', fontsize=13)
        ax.set_ylabel(f'{stat.capitalize()} hits per doc' if ax == axes[0] else '', fontsize=11)
        ax.set_ylim(bottom=0)
        ax.grid(axis='y', alpha=0.3)
        ax.set_xticks(range(2005, max_year + 1, 2))
        ax.set_xlim(2004.5, max_year + 0.5)
        ax.tick_params(axis='x', rotation=45, labelsize=10)
        ax.axvline(x=2017.5, color='red', linestyle=':', alpha=0.4, linewidth=1)
        ax.legend(fontsize=11, loc='best')
    
    fig.suptitle(f'{topic_label} — All Doc Types, by {group_label}\nHits per doc (country-year) with IQR',
                 fontweight='bold', fontsize=14, y=1.02)
    
    fig.text(0.05, -0.06,
             'Note: Each country contributes one value per year (avg hits per doc across DSA + AIV + Programs). '
             'Shaded band = IQR (P25–P75).\nRed dotted line = 2018 LIC-DSF revision. '
             'Source: IEO Staff\'s calculations from IMF documents.',
             fontsize=9, ha='left')
    
    plt.tight_layout()
    
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        safe_group = group_col.lower().replace(' ', '_')
        fname = os.path.join(save_dir, f'iqr_alldocs_{safe_group}_{topic_key}.png')
        fig.savefig(fname, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    else:
        plt.show()

### 4e. Bucket LICs by Document Type

In [16]:
# % of DOCUMENTS MENTIONS PER TYPE DOCUMENT PER TOPIC 
def plot_bucket_lics_by_doctype(df_all, bucket_name, topics, max_year=2023, save_dir=None):
    """Three panels side by side: one per doc type."""
    def smooth(series):
        return series.rolling(3, center=True, min_periods=2).mean()
    
    fig, axes = plt.subplots(1, 3, figsize=(24, 6), sharey=True)
    
    for ax, doc_type in zip(axes, ['DSA', 'AIV', 'Program']):
        df_sub = df_all[(df_all['doc_type'] == doc_type) & (df_all['year'] <= max_year)]
        
        has_cols = [f'has_{tk}' for tk, _ in topics]
        yearly = df_sub.groupby('year').agg(
            n_docs=('filename', 'count'),
            **{col: (col, 'sum') for col in has_cols}
        ).reset_index()
        for tk, _ in topics:
            yearly[f'pct_{tk}'] = yearly[f'has_{tk}'] / yearly['n_docs'] * 100
        
        for i, (tk, tl) in enumerate(topics):
            color = GRID_COLORS[i % len(GRID_COLORS)]
            marker = GRID_MARKERS[i % len(GRID_MARKERS)]
            ax.plot(yearly['year'], smooth(yearly[f'pct_{tk}']), '-', marker=marker,
                    linewidth=2.5, color=color, markersize=5, zorder=3, label=tl)
        
        ax2 = ax.twinx()
        ax2.bar(yearly['year'], yearly['n_docs'], color='gray', alpha=0.1, width=0.8, zorder=0)
        ax2.set_ylim(0, yearly['n_docs'].max() * 3)
        ax2.tick_params(axis='y', labelsize=8, colors='gray')
        if ax == axes[-1]:
            ax2.set_ylabel('N docs', fontsize=10, color='gray', alpha=0.5)
        
        ax.set_title(f'{doc_type}', fontweight='bold', fontsize=13)
        ax.set_ylabel('% of documents' if ax == axes[0] else '', fontsize=11)
        ax.set_ylim(bottom=0)
        ax.grid(axis='y', alpha=0.3)
        ax.set_xticks(range(2005, max_year + 1, 2))
        ax.set_xlim(2004.5, max_year + 0.5)
        ax.tick_params(axis='x', rotation=45, labelsize=10)
        ax.axvline(x=2017.5, color='red', linestyle=':', alpha=0.4, linewidth=1)
        ax.legend(fontsize=9, loc='best', frameon=True)
    
    fig.suptitle(f'{bucket_name} — All LICs, by Document Type\n'
                 '% of documents with at least one mention',
                 fontweight='bold', fontsize=14, y=1.02)
    
    fig.text(0.05, -0.04,
             'Note: 3-year centered rolling average. Red dotted line = 2018 LIC-DSF revision. '
             'Gray bars = documents per year.\n'
             'Source: IEO Staff\'s calculations from IMF documents.',
             fontsize=9, ha='left')
    plt.tight_layout()
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        safe_name = bucket_name.lower().replace(' ', '_').replace('/', '_').replace('&', 'and')
        fname = os.path.join(save_dir, f'by_doctype_{safe_name}.png')
        fig.savefig(fname, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    else:
        plt.show()


# Run
soe_clean_list = [
    ('soes_clean', 'SOEs'),
    ('debt_transparency_clean', 'Debt Transparency'),
    ('contingent_liabilities_clean', 'Contingent Liabilities'),
    ('data_quality_clean', 'Data Quality'),
]

plot_bucket_lics_by_doctype(df_clean_all, 'SOE & Transparency (cleaned)', 
                             soe_clean_list, max_year=2023)

NameError: name 'df_clean_all' is not defined

### 4f. Keyword Mentions Compare (Country Groupings)

In [29]:
def plot_topic_iqr_by_grouping(df_mentions_all, topic_key, topic_label, grouping,
                                max_year=2023, save_dir=None, doc_type='DSA'):
    """
    Side by side: median with IQR (left) vs mean with IQR (right).
    Two lines per panel: group=1 vs group=0.
    Uses normalized counts (per 100 words).
    """
    group_labels = {
        'hipc': ('HIPC', 'Non-HIPC'),
        'fcs': ('FCS', 'Non-FCS'),
        'cex': ('Commodity Exporters', 'Non-CEX'),
        'sds': ('Small Developing States', 'Non-SDS'),
        'rst': ('RST', 'Non-RST'),
        'frontier_market': ('Frontier Markets', 'Non-Frontier'),
        'exposure_china': ('China Exposure', 'No China Exposure'),
    }
    yes_label, no_label = group_labels.get(grouping, (f'{grouping}=1', f'{grouping}=0'))

    df = df_mentions_all[(df_mentions_all['doc_type'] == doc_type)
                          & (df_mentions_all['year'] <= max_year)].copy()

    col = f'count_{topic_key}_per100w'
    colors = {1: 'tab:blue', 0: 'tab:orange'}
    labels = {1: yes_label, 0: no_label}

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    for ax, stat in zip(axes, ['median', 'mean']):
        for val in [1, 0]:
            sub = df[df[grouping] == val]
            n_countries = sub['ifs'].nunique()

            cy = sub.groupby(['country', 'year']).agg(
                avg_hits=(col, 'mean')
            ).reset_index()

            yearly = cy.groupby('year')['avg_hits'].agg(
                center=stat,
                p25=lambda x: x.quantile(0.25),
                p75=lambda x: x.quantile(0.75),
            ).reset_index()

            ax.fill_between(yearly['year'], yearly['p25'], yearly['p75'],
                             alpha=0.12, color=colors[val])
            ax.plot(yearly['year'], yearly['center'], '-o', linewidth=2.5,
                    color=colors[val], markersize=5, zorder=3,
                    label=f'{labels[val]} (n={n_countries})')

        ax.set_title(f'{stat.capitalize()}', fontweight='bold', fontsize=13)
        ax.set_ylabel(f'{stat.capitalize()} hits per 100 words', fontsize=11)
        ax.set_ylim(bottom=0)
        ax.grid(axis='y', alpha=0.3)
        ax.set_xticks(range(2005, max_year + 1, 2))
        ax.set_xlim(2004.5, max_year + 0.5)
        ax.tick_params(axis='x', rotation=45, labelsize=10)
        ax.axvline(x=2017.5, color='red', linestyle=':', alpha=0.4, linewidth=1)
        ax.legend(fontsize=10)

    fig.suptitle(f'{topic_label} — {yes_label} vs {no_label} ({doc_type} only)\n'
                 f'Keyword hits per 100 words (country-year level)',
                 fontweight='bold', fontsize=14, y=1.02)

    from textwrap import fill
    note_text = (
        f'Note: Each country contributes one observation per year, computed as the average keyword hits '
        f'per 100 words within that country-year. Left panel shows the median with IQR band (P25–P75); '
        f'right panel shows the mean with IQR band. Blue = {yes_label}; Orange = {no_label}. '
        f'Red dotted line marks the 2018 LIC-DSF revision. '
        f'Source: IEO staff calculations based on IMF {doc_type} documents.'
    )
    fig.text(0.05, -0.06, fill(note_text, width=150), fontsize=9, ha='left', va='top')

    plt.tight_layout()

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fname = os.path.join(save_dir, f'iqr_{grouping}_{topic_key}_{doc_type}.png')
        fig.savefig(fname, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    else:
        plt.show()

In [ ]:
# DSA
for grouping in ['cex', 'fcs', 'sds', 'hipc', 'frontier_market', 'exposure_china']:
    for tk, tcfg in MENTION_TOPICS.items():
        plot_topic_iqr_by_grouping(df_mentions_all, tk, tcfg['description'],
                                    grouping, max_year=2023, doc_type='DSA')

### 4g. Country-Level Mentions

In [ ]:
from matplotlib.ticker import MaxNLocator

def plot_country_mentions(df_mentions_all, country_name, max_year=2024):
    """
    Plot keyword mentions for a single country (DSA only).
    3 figures × 3 panels. Uses count_X (hit counts), no smoothing.
    """
    df = df_mentions_all[(df_mentions_all['doc_type'] == 'DSA')
                          & (df_mentions_all['country'] == country_name)
                          & (df_mentions_all['year'] <= max_year)]

    if len(df) == 0:
        print(f"No DSA documents found for '{country_name}'")
        return

    count_cols = [c for c in df.columns if c.startswith('count_')]
    yearly = df.groupby('year').agg(
        n_docs=('filename', 'count'),
        **{col: (col, 'sum') for col in count_cols}
    ).reset_index()

    n_docs = len(df)
    years_range = f"{int(df['year'].min())}-{int(df['year'].max())}"

    figures = [
        ('DSA & Debt Analysis', [
            ('DSA Tools', [
                ('market_financing_tool', 'Market Financing Tool', 'o', 2.5, 'tab:blue'),
                ('probability_approach', 'Probability Approach', 's', 2.5, 'tab:orange'),
            ], True),
            ('DSA Framework', [
                ('realism_tools',          'Realism Tools',          'o', 2.5, 'tab:blue'),
                ('market_access',          'Market Access',          's', 2.5, 'tab:orange'),
                ('contingent_liabilities', 'Contingent Liabilities', '^', 1.8, 'tab:green'),
                ('creditor_composition',   'Creditor Composition',   'D', 1.8, 'tab:red'),
                ('debt_restructuring',     'Debt Restructuring',     'v', 1.8, 'tab:purple'),
            ], True),
            ('Debt Distress Indicators', [
                ('solvency_issues',  'Solvency',   'o', 2.5, 'tab:blue'),
                ('liquidity_issues', 'Liquidity',  's', 2.5, 'tab:orange'),
                ('arrears',          'Arrears',    '^', 2.5, 'tab:red'),
                ('domestic_debt',    'Domestic Debt',  'D', 1.8, 'tab:purple'),
            ], False),
        ]),
        ('Creditor Landscape', [
            ('Traditional Bilateral', [
                ('paris_club',      'Paris Club',      'o', 2.5, 'tab:blue'),
                ('france',          'France',          's', 1.8, 'tab:cyan'),
                ('japan',           'Japan',           '^', 1.8, 'tab:red'),
                ('united_kingdom',  'United Kingdom',  'x', 1.2, 'tab:orange'),
            ], False),
            ('Non-Traditional Bilateral', [
                ('china',       'China',       'o', 2.5, 'tab:blue'),
                ('india',       'India',       's', 1.8, 'tab:orange'),
                ('gulf_states', 'Gulf States', '^', 1.8, 'tab:green'),
                ('russia',      'Russia',      'D', 1.8, 'tab:red'),
            ], False),
            ('Creditor Groups', [
                ('paris_club',       'Paris Club',         'o', 2.5, 'tab:blue'),
                ('non_paris_club',   'Non-Paris Club',     's', 2.5, 'tab:red'),
                ('ifi_multilateral', 'IFI / Multilateral', '^', 2.5, 'tab:green'),
            ], True),
        ]),
        ('Structural & Policy', [
            ('Structural', [
                ('debt_investment_nexus', 'Debt-Investment Nexus', 'o', 2.5, 'tab:blue'),
                ('external_shocks',      'External Shocks',       's', 1.8, 'tab:orange'),
                ('domestic_conditions',  'Domestic Conditions',   '^', 1.8, 'tab:green'),
                ('remittances',          'Remittances',           'D', 1.8, 'tab:red'),
            ], False),
            ('Policy & Management', [
                ('public_debt_management',     'Debt Management', 'o', 2.5, 'tab:blue'),
                ('non_concessional_borrowing', 'NCB',             's', 2.5, 'tab:orange'),
                ('data_quality',               'Data Quality',    '^', 1.8, 'tab:green'),
            ], False),
            ('Market / Private', [
                ('private_creditors', 'Private/Commercial',  'o', 2.5, 'tab:blue'),
                ('bondholders',       'Bondholders/Bonds',   's', 2.5, 'tab:orange'),
                ('rating_agencies',   'Rating Agencies',     '^', 1.2, 'tab:gray'),
            ], False),
        ]),
    ]

    for fig_title, panels in figures:
        fig, axes = plt.subplots(1, 3, figsize=(20, 5))

        for ax, (panel_title, topics, show_vline) in zip(axes, panels):
            for topic, label, marker, lw, color in topics:
                col = f'count_{topic}'
                if col not in yearly.columns:
                    continue
                ax.plot(yearly['year'], yearly[col], '-', marker=marker, linewidth=lw,
                        label=label, color=color, markersize=6, zorder=3)

            if show_vline:
                ax.axvline(x=2017.5, color='red', linestyle=':', alpha=0.5, linewidth=1)
            ax.set_title(panel_title, fontweight='bold', fontsize=11)
            ax.legend(fontsize=7, ncol=2)

        for ax in axes:
            ax.set_xlabel('Year', fontsize=9)
            ax.set_ylabel('Keyword hits', fontsize=9)
            ax.set_ylim(bottom=0)
            y_max = max(int(ax.get_ylim()[1]) + 1, 2)
            ax.set_ylim(0, y_max)
            if y_max <= 10:
                ax.set_yticks(range(0, y_max + 1))
            elif y_max <= 30:
                ax.set_yticks(range(0, y_max + 1, 5))
            else:
                ax.set_yticks(range(0, y_max + 1, 10))
            ax.grid(axis='y', alpha=0.3)
            ax.set_xticks(range(2005, max_year + 1, 2))
            ax.set_xlim(2004.5, max_year + 0.5)
            ax.tick_params(axis='x', rotation=45, labelsize=8)

            ax2 = ax.twinx()
            ax2.bar(yearly['year'], yearly['n_docs'], color='gray', alpha=0.12, width=0.8, zorder=0)
            ax2.set_ylabel('N docs', fontsize=8, color='gray', alpha=0.5)
            n_max = max(int(yearly['n_docs'].max()) + 1, 2)
            ax2.set_ylim(0, n_max)
            ax2.set_yticks(range(0, n_max + 1))
            ax2.tick_params(axis='y', labelsize=6, colors='gray')

        fig.suptitle(f'{fig_title} — {country_name}\n'
                     f'{n_docs} DSA documents ({years_range})',
                     fontsize=13, fontweight='bold', y=1.05)

        fig.text(0.01, -0.06,
                 'Note: Y-axis = total keyword hits per year (summed if multiple DSAs). '
                 'Gray bars = number of DSAs.\n'
                 'Source: IEO Staff\'s calculations from IMF DSA documents.',
                 fontsize=8, ha='left')

        plt.tight_layout()
        plt.subplots_adjust(bottom=0.08)
        plt.show()

## 5. Generate Charts

Execution cells for producing the paper's figures.

### 5a. Bucket Grids

### 5b. IQR Comparison Plots

#### Graphs RAW

In [27]:
SAVE_DIR_IQR = str(OUTPUT_DIR_CHARTS / 'iqr_comparison')

for doc_type in ['DSA', 'AIV', 'Program']:
    for normalized in [False, True]:
        tag = 'normalized' if normalized else 'raw'
        sub_dir = os.path.join(SAVE_DIR_IQR, doc_type.lower(), tag)
        os.makedirs(sub_dir, exist_ok=True)
        for tk, tcfg in MENTION_TOPICS.items():
            plot_topic_iqr_comparison(df_mentions_all, tk, tcfg['description'],
                                      max_year=2023, save_dir=sub_dir,
                                      doc_type=doc_type, normalized=normalized)
        print(f"✓ {doc_type} {tag} done")

✓ DSA raw done
✓ DSA normalized done
✓ AIV raw done
✓ AIV normalized done
✓ Program raw done
✓ Program normalized done


In [ ]:
# Cell A — DSA
for tk, tcfg in MENTION_TOPICS.items():
    plot_topic_iqr_comparison(df_mentions_all, tk, tcfg['description'],
                              max_year=2023, doc_type='DSA')

In [ ]:
# Cell B — AIV
for tk, tcfg in MENTION_TOPICS.items():
    plot_topic_iqr_comparison(df_mentions_all, tk, tcfg['description'],
                              max_year=2023, doc_type='AIV')

In [ ]:
# Cell C — Program
for tk, tcfg in MENTION_TOPICS.items():
    plot_topic_iqr_comparison(df_mentions_all, tk, tcfg['description'],
                              max_year=2023, doc_type='Program')

#### Graphs normalizadas

In [ ]:
# Normalizar: hits por cada 100 palabras
for tk in MENTION_TOPICS:
    col = f'count_{tk}'
    df_mentions_all[f'{col}_per100w'] = (df_mentions_all[col] / df_mentions_all['word_count']) * 100
print(f"✓ Normalized columns added: {len(MENTION_TOPICS)} topics")

In [16]:
def plot_topic_iqr_comparison(df_mentions_all, topic_key, topic_label, 
                               max_year=2023, save_dir=None, doc_type='DSA',
                               normalized=False):
    """Side by side: median with IQR (left) vs mean without band (right). Independent y-axes."""
    
    df = df_mentions_all[(df_mentions_all['doc_type'] == doc_type)
                          & (df_mentions_all['year'] <= max_year)].copy()
    
    col = f'count_{topic_key}_per100w' if normalized else f'count_{topic_key}'
    ylabel_suffix = 'per 100 words' if normalized else 'per doc'
    norm_tag = '_norm' if normalized else ''
    
    cy = df.groupby(['country', 'year']).agg(
        avg_hits=(col, 'mean')
    ).reset_index()
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    
    for ax, stat in zip(axes, ['median', 'mean']):
        yearly = cy.groupby('year')['avg_hits'].agg(
            center=stat,
            p25=lambda x: x.quantile(0.25),
            p75=lambda x: x.quantile(0.75),
        ).reset_index()
        
        if stat == 'median':
            ax.fill_between(yearly['year'], yearly['p25'], yearly['p75'],
                             alpha=0.2, color='tab:blue')
        
        ax.plot(yearly['year'], yearly['center'], '-o', linewidth=2.5,
                color='tab:blue', markersize=5, zorder=3)
        
        ax.set_title(f'{stat.capitalize()}', fontweight='bold', fontsize=13)
        ax.set_ylabel(f'{stat.capitalize()} hits {ylabel_suffix}', fontsize=11)
        ax.set_ylim(bottom=0)
        ax.grid(axis='y', alpha=0.3)
        ax.set_xticks(range(2005, max_year + 1, 2))
        ax.set_xlim(2004.5, max_year + 0.5)
        ax.tick_params(axis='x', rotation=45, labelsize=10)
        ax.axvline(x=2017.5, color='red', linestyle=':', alpha=0.4, linewidth=1)
    
    fig.suptitle(f'{topic_label} — All LICs ({doc_type} only)\n'
                 f'Keyword hits {ylabel_suffix} (country-year level)',
                 fontweight='bold', fontsize=14, y=1.02)
    
    from textwrap import fill
    note_text = (
        'Note: Each country contributes one observation per year, computed as the average keyword hits '
        f'{ylabel_suffix} within that country-year. Left panel shows the median across countries with IQR band '
        '(P25–P75); right panel shows the mean. The median captures the typical country experience, '
        'while the mean reflects overall intensity including outliers. '
        'Red dotted line marks the 2018 LIC-DSF revision. '
        'DSA = Debt Sustainability Analysis; LIC-DSF = Low-Income Country Debt Sustainability Framework; '
        'IQR = Interquartile Range. '
        'Source: IEO staff calculations based on IMF DSA documents.'
    )
    fig.text(0.05, -0.06, fill(note_text, width=150), fontsize=9, ha='left', va='top')
    
    plt.tight_layout()
    
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fname = os.path.join(save_dir, f'iqr_comparison_{topic_key}_{doc_type}{norm_tag}.png')
        fig.savefig(fname, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close(fig)
    else:
        plt.show()

In [ ]:
# DSA normalized
for tk, tcfg in MENTION_TOPICS.items():
    plot_topic_iqr_comparison(df_mentions_all, tk, tcfg['description'],
                              max_year=2023, doc_type='DSA', normalized=True)

In [ ]:
# AIV normalized
for tk, tcfg in MENTION_TOPICS.items():
    plot_topic_iqr_comparison(df_mentions_all, tk, tcfg['description'],
                              max_year=2023, doc_type='AIV', normalized=True)

In [ ]:
# Program Raw
for tk, tcfg in MENTION_TOPICS.items():
    plot_topic_iqr_comparison(df_mentions_all, tk, tcfg['description'],
                              max_year=2023, doc_type='Program', normalized=False)

In [ ]:
# Program normalized
for tk, tcfg in MENTION_TOPICS.items():
    plot_topic_iqr_comparison(df_mentions_all, tk, tcfg['description'],
                              max_year=2023, doc_type='Program', normalized=True)

In [17]:
# Saving RAW and NORMALIZED Program
SAVE_DIR_IQR = str(OUTPUT_DIR_CHARTS / 'iqr_comparison')
for normalized in [False, True]:
    tag = 'normalized' if normalized else 'raw'
    sub_dir = os.path.join(SAVE_DIR_IQR, 'program', tag)
    os.makedirs(sub_dir, exist_ok=True)
    for tk, tcfg in MENTION_TOPICS.items():
        plot_topic_iqr_comparison(df_mentions_all, tk, tcfg['description'],
                                  max_year=2023, save_dir=sub_dir,
                                  doc_type='Program', normalized=normalized)
    print(f"✓ Program {tag} done")

✓ Program raw done
✓ Program normalized done


In [ ]:
dsa_per_year = df.groupby('year').size().reset_index(name='n_dsa')
print(dsa_per_year.to_string(index=False))

### 5c0 Export dataframes

##### Extract month AIV

In [20]:
# Cell: GPU check
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    total_gb = gpu.total_memory / 1024**3
    alloc_gb = torch.cuda.memory_allocated(0) / 1024**3
    reserved_gb = torch.cuda.memory_reserved(0) / 1024**3
    free_gb = total_gb - reserved_gb
    
    print(f"GPU: {gpu.name}")
    print(f"Total VRAM:     {total_gb:.1f} GB")
    print(f"Allocated:      {alloc_gb:.1f} GB")
    print(f"Reserved:       {reserved_gb:.1f} GB")
    print(f"Free (approx):  {free_gb:.1f} GB")
    print(f"\nQwen2.5-7B needs ~14 GB. {'✓ Enough VRAM' if free_gb > 14 else '⚠ Might be tight'}")
else:
    print("⚠ No CUDA GPU detected")

GPU: Tesla V100-PCIE-32GB
Total VRAM:     31.9 GB
Allocated:      0.0 GB
Reserved:       0.0 GB
Free (approx):  31.9 GB

Qwen2.5-7B needs ~14 GB. ✓ Enough VRAM


In [21]:
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
CACHE_DIR = r'E:/Data/gsoto/hf_cache'

qwen_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="cuda:0",
    trust_remote_code=True, cache_dir=CACHE_DIR)
qwen_tokenizer = AutoProcessor.from_pretrained(
    MODEL_ID, trust_remote_code=True, cache_dir=CACHE_DIR)
print(f"✓ Loaded | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

E:\Data\gsoto\venvs\venv-bp2\Lib\site-packages\transformers\models\auto\modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


✓ Loaded | VRAM: 15.4 GB


In [22]:
# Cell: Extract AIV months — review stage (3000 chars)
aiv_mask = df_mentions_all['doc_type'] == 'AIV'
filepaths = df_aiv.set_index('filename')['filepath']

review_rows = []

for _, row in tqdm(df_mentions_all[aiv_mask].iterrows(), total=aiv_mask.sum(), desc='AIV months'):
    fname = row['filename']
    fp = Path(filepaths.get(fname, ''))
    
    if not fp.exists():
        review_rows.append({'filename': fname, 'year': row['year'], 'month_extracted': None, 'text_snippet': 'FILE NOT FOUND'})
        continue
    
    try:
        text = fp.read_text(encoding='utf-8')[:3000]
    except:
        text = fp.read_text(encoding='latin-1')[:3000]
    
    messages = [
        {"role": "system", "content": "You extract document dates. Return ONLY a number 1-12 for the month. Nothing else."},
        {"role": "user", "content": f"What month was this document published or discussed? Return only the month number.\n\n{text}"}
    ]
    
    input_text = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_tokenizer(text=input_text, return_tensors="pt").to("cuda:0")
    
    with torch.no_grad():
        output = qwen_model.generate(**inputs, max_new_tokens=5, temperature=0.1, do_sample=False)
    
    response = qwen_tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    
    try:
        month = int(response)
        if not 1 <= month <= 12:
            month = None
            print(f"  ⚠ {fname}: got {response}")
    except:
        month = None
        print(f"  ⚠ {fname}: got '{response}'")
    
    review_rows.append({
        'filename': fname,
        'year': row['year'],
        'month_extracted': month,
        'text_snippet': text, #[:500],
    })

df_aiv_months = pd.DataFrame(review_rows)
print(f"\nExtracted: {df_aiv_months['month_extracted'].notna().sum()}/{len(df_aiv_months)}")
print(f"Failed: {df_aiv_months['month_extracted'].isna().sum()}")

AIV months: 100%|████| 733/733 [03:43<00:00,  3.28it/s]


Extracted: 733/733
Failed: 0


In [25]:
# Cell: Map to df_mentions_all (run only after review)
aiv_mask = df_mentions_all['doc_type'] == 'AIV'
month_map = df_aiv_months.set_index('filename')['month_extracted']
df_mentions_all.loc[aiv_mask, 'month'] = df_mentions_all.loc[aiv_mask, 'filename'].map(month_map)

print(f"AIV months mapped: {df_mentions_all.loc[aiv_mask, 'month'].notna().sum()}/{aiv_mask.sum()}")

AIV months mapped: 733/733


In [ ]:
del qwen_model, qwen_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [27]:
dsa = df_mentions_all[df_mentions_all['doc_type'] == 'DSA'][['filename', 'ifs', 'country', 'year', 'month']].copy()
aiv = df_mentions_all[df_mentions_all['doc_type'] == 'AIV'][['filename', 'ifs', 'country', 'year', 'month']].copy()
prog = df_mentions_all[df_mentions_all['doc_type'] == 'Program'][['filename', 'ifs', 'country', 'year', 'month']].copy()

##### Month DSA y Programs

In [28]:
month_map = {'january':1, 'february':2, 'march':3, 'april':4, 'may':5, 'june':6,
             'july':7, 'august':8, 'september':9, 'october':10, 'november':11, 'december':12, 'decembre':12}

# DSA: extract from filename
dsa_mask = df_mentions_all['doc_type'] == 'DSA'
df_mentions_all.loc[dsa_mask, 'month'] = (
    df_mentions_all.loc[dsa_mask, 'filename']
    .str.split('_').str[-1]
    .str.lower()
    .map(month_map)
)

print(f"DSA months: {df_mentions_all.loc[dsa_mask, 'month'].notna().sum()}/{dsa_mask.sum()}")
print(df_mentions_all.loc[dsa_mask, ['filename', 'year', 'month']].head(10))

DSA months: 924/924
                    filename  year  month
0      Afghanistan_2007_June  2007    6.0
1   Afghanistan_2010_January  2010    1.0
2  Afghanistan_2011_November  2011   11.0
3      Afghanistan_2012_June  2012    6.0
4     Afghanistan_2014_April  2014    4.0
5  Afghanistan_2015_November  2015   11.0
6      Afghanistan_2016_July  2016    7.0
7       Afghanistan_2017_May  2017    5.0
8  Afghanistan_2017_November  2017   11.0
9  Afghanistan_2018_December  2018   12.0


In [29]:
prog_mask = df_mentions_all['doc_type'] == 'Program'

meta = pd.read_excel(r'U:\dil\input\clean_meta_complete.xlsx',
                     sheet_name='Final_Final',
                     usecols=['File_Name', 'Final_Document_Date'])
meta['Final_Document_Date'] = pd.to_datetime(meta['Final_Document_Date'], format='mixed')
meta['month'] = meta['Final_Document_Date'].dt.month

prog_filename_clean = df_mentions_all.loc[prog_mask, 'filename'].str.replace('.md', '', regex=False)
month_mapping = meta.set_index('File_Name')['month']

df_mentions_all.loc[prog_mask, 'month'] = prog_filename_clean.map(month_mapping).values

print(f"Program months: {df_mentions_all.loc[prog_mask, 'month'].notna().sum()}/{prog_mask.sum()}")

Program months: 944/944


##### Build df_export

###### Pairing DSA AIV Program

In [60]:
for dt in ['DSA', 'AIV', 'Program']:
    n = (df_mentions_all['doc_type'] == dt).sum()
    print(f"{dt}: {n}")
print(f"Total: {len(df_mentions_all)}")

DSA: 924
AIV: 733
Program: 944
Total: 2601


In [51]:
df_mentions_all['paired_conditional'] = None
df_mentions_all['paired_conditional_filename'] = None
print("✓ Reset")

✓ Reset


In [52]:
# Pairing
# Build pairing: Program priority, then AIV, ±2 months
dsa_pairs = {}

for _, d in tqdm(dsa.iterrows(), total=len(dsa), desc='Pairing DSAs'):
    fname = d['filename']
    
    # Try Program first
    match_p = prog[(prog['ifs'] == d['ifs']) & (prog['year'] == d['year']) & 
                   (prog['month'].between(d['month']-2, d['month']+2))]
    if len(match_p) > 0:
        # Pick closest month
        best = match_p.iloc[(match_p['month'] - d['month']).abs().argsort().iloc[0]]
        dsa_pairs[fname] = {'paired_with': 'Program', 'paired_filename': best['filename']}
        continue
    
    # Try AIV
    match_a = aiv[(aiv['ifs'] == d['ifs']) & (aiv['year'] == d['year']) & 
                  (aiv['month'].between(d['month']-2, d['month']+2))]
    if len(match_a) > 0:
        best = match_a.iloc[(match_a['month'] - d['month']).abs().argsort().iloc[0]]
        dsa_pairs[fname] = {'paired_with': 'AIV', 'paired_filename': best['filename']}
        continue
    
    dsa_pairs[fname] = {'paired_with': None, 'paired_filename': None}

# Map to df_mentions_all
dsa_mask = df_mentions_all['doc_type'] == 'DSA'
df_mentions_all.loc[dsa_mask, 'paired_with'] = df_mentions_all.loc[dsa_mask, 'filename'].map(
    lambda x: dsa_pairs.get(x, {}).get('paired_with'))
df_mentions_all.loc[dsa_mask, 'paired_filename'] = df_mentions_all.loc[dsa_mask, 'filename'].map(
    lambda x: dsa_pairs.get(x, {}).get('paired_filename'))

# Summary
paired = df_mentions_all.loc[dsa_mask, 'paired_with'].value_counts(dropna=False)
print(paired)

Pairing DSAs: 100%|█| 924/924 [00:00<00:00, 1040.61it/s

paired_with
Program    458
AIV        289
None       177
Name: count, dtype: int64


In [53]:
# Cell: Conditional pairing for unmatched DSAs (same year, DSA month ≤ parent month, closest)
unmatched_mask = dsa_mask & df_mentions_all['paired_with'].isna()
unmatched_fnames = df_mentions_all.loc[unmatched_mask, 'filename'].tolist()

conditional_pairs = {}

for _, d in tqdm(dsa[dsa['filename'].isin(unmatched_fnames)].iterrows(), 
                 total=len(unmatched_fnames), desc='Conditional pairing'):
    fname = d['filename']
    
    # Program first: same country + year, DSA month ≤ Program month
    match_p = prog[(prog['ifs'] == d['ifs']) & (prog['year'] == d['year']) & 
                   (prog['month'] >= d['month'])]
    if len(match_p) > 0:
        best = match_p.iloc[(match_p['month'] - d['month']).abs().argsort().iloc[0]]
        conditional_pairs[fname] = {'paired_conditional': 'Program', 'paired_conditional_filename': best['filename']}
        continue
    
    # AIV: same logic
    match_a = aiv[(aiv['ifs'] == d['ifs']) & (aiv['year'] == d['year']) & 
                  (aiv['month'] >= d['month'])]
    if len(match_a) > 0:
        best = match_a.iloc[(match_a['month'] - d['month']).abs().argsort().iloc[0]]
        conditional_pairs[fname] = {'paired_conditional': 'AIV', 'paired_conditional_filename': best['filename']}
        continue
    
    conditional_pairs[fname] = {'paired_conditional': None, 'paired_conditional_filename': None}

# Map to df_mentions_all
df_mentions_all['paired_conditional'] = None
df_mentions_all['paired_conditional_filename'] = None

df_mentions_all.loc[unmatched_mask, 'paired_conditional'] = df_mentions_all.loc[unmatched_mask, 'filename'].map(
    lambda x: conditional_pairs.get(x, {}).get('paired_conditional'))
df_mentions_all.loc[unmatched_mask, 'paired_conditional_filename'] = df_mentions_all.loc[unmatched_mask, 'filename'].map(
    lambda x: conditional_pairs.get(x, {}).get('paired_conditional_filename'))

# Summary
cond = df_mentions_all.loc[unmatched_mask, 'paired_conditional'].value_counts(dropna=False)
print(f"\nConditional pairing (187 previously unmatched):")
print(cond)

Conditional pairing: 100%|█| 177/177 [00:00<00:00, 1161


Conditional pairing (187 previously unmatched):
paired_conditional
None       125
Program     34
AIV         18
Name: count, dtype: int64


In [54]:
# Cell: Second conditional pairing — same year, any month, closest
still_unmatched_mask = dsa_mask & df_mentions_all['paired_with'].isna() & df_mentions_all['paired_conditional'].isna()
still_unmatched_fnames = df_mentions_all.loc[still_unmatched_mask, 'filename'].tolist()

conditional_pairs_2 = {}

for _, d in tqdm(dsa[dsa['filename'].isin(still_unmatched_fnames)].iterrows(),
                 total=len(still_unmatched_fnames), desc='Conditional pairing 2'):
    fname = d['filename']
    
    match_p = prog[(prog['ifs'] == d['ifs']) & (prog['year'] == d['year'])]
    if len(match_p) > 0:
        best = match_p.iloc[(match_p['month'] - d['month']).abs().argsort().iloc[0]]
        conditional_pairs_2[fname] = {'paired_conditional': 'Program', 'paired_conditional_filename': best['filename']}
        continue
    
    match_a = aiv[(aiv['ifs'] == d['ifs']) & (aiv['year'] == d['year'])]
    if len(match_a) > 0:
        best = match_a.iloc[(match_a['month'] - d['month']).abs().argsort().iloc[0]]
        conditional_pairs_2[fname] = {'paired_conditional': 'AIV', 'paired_conditional_filename': best['filename']}
        continue
    
    conditional_pairs_2[fname] = {'paired_conditional': None, 'paired_conditional_filename': None}

# Map
df_mentions_all.loc[still_unmatched_mask, 'paired_conditional'] = df_mentions_all.loc[still_unmatched_mask, 'filename'].map(
    lambda x: conditional_pairs_2.get(x, {}).get('paired_conditional'))
df_mentions_all.loc[still_unmatched_mask, 'paired_conditional_filename'] = df_mentions_all.loc[still_unmatched_mask, 'filename'].map(
    lambda x: conditional_pairs_2.get(x, {}).get('paired_conditional_filename'))

cond2 = df_mentions_all.loc[still_unmatched_mask, 'paired_conditional'].value_counts(dropna=False)
print(f"\nConditional pairing 2 (139 previously unmatched):")
print(cond2)

Conditional pairing 2: 100%|█| 125/125 [00:00<00:00, 15


Conditional pairing 2 (139 previously unmatched):
paired_conditional
None       93
AIV        18
Program    14
Name: count, dtype: int64


In [55]:
# Conditional 3 — year-1 AND year+1, pick closest in absolute time
still_unmatched_mask = dsa_mask & df_mentions_all['paired_with'].isna() & df_mentions_all['paired_conditional'].isna()
still_unmatched_fnames = df_mentions_all.loc[still_unmatched_mask, 'filename'].tolist()

conditional_pairs_3 = {}

for _, d in tqdm(dsa[dsa['filename'].isin(still_unmatched_fnames)].iterrows(),
                 total=len(still_unmatched_fnames), desc='Conditional pairing 3'):
    fname = d['filename']
    dsa_time = d['year'] * 12 + d['month']
    
    # Collect all candidates from year-1 and year+1
    candidates = []
    
    for delta in [-1, 1]:
        target_year = d['year'] + delta
        
        for source, doc_type in [(prog, 'Program'), (aiv, 'AIV')]:
            matches = source[(source['ifs'] == d['ifs']) & (source['year'] == target_year)]
            for _, m in matches.iterrows():
                m_time = m['year'] * 12 + m['month']
                candidates.append({
                    'type': doc_type,
                    'filename': m['filename'],
                    'distance': abs(dsa_time - m_time),
                })
    
    if candidates:
        # Pick closest, prioritize Program if tied
        candidates.sort(key=lambda x: (x['distance'], 0 if x['type'] == 'Program' else 1))
        best = candidates[0]
        conditional_pairs_3[fname] = {'paired_conditional': best['type'], 'paired_conditional_filename': best['filename']}
    else:
        conditional_pairs_3[fname] = {'paired_conditional': None, 'paired_conditional_filename': None}

df_mentions_all.loc[still_unmatched_mask, 'paired_conditional'] = df_mentions_all.loc[still_unmatched_mask, 'filename'].map(
    lambda x: conditional_pairs_3.get(x, {}).get('paired_conditional'))
df_mentions_all.loc[still_unmatched_mask, 'paired_conditional_filename'] = df_mentions_all.loc[still_unmatched_mask, 'filename'].map(
    lambda x: conditional_pairs_3.get(x, {}).get('paired_conditional_filename'))

cond3 = df_mentions_all.loc[still_unmatched_mask, 'paired_conditional'].value_counts(dropna=False)
print(f"\nConditional pairing 3 — closest year±1 ({len(still_unmatched_fnames)} unmatched):")
print(cond3)

Conditional pairing 3: 100%|█| 93/93 [00:00<00:00, 698.


Conditional pairing 3 — closest year±1 (93 unmatched):
paired_conditional
AIV        64
Program    26
None        3
Name: count, dtype: int64


In [56]:
final_unmatched = df_mentions_all.loc[
    dsa_mask & df_mentions_all['paired_with'].isna() & df_mentions_all['paired_conditional'].isna(),
    ['country', 'year', 'month', 'filename']]
print(f"Still unmatched: {len(final_unmatched)}")
print(final_unmatched.head(20))

Still unmatched: 3
                            country  year  month  \
190                         Comoros  2021   10.0   
378                           Haiti  2022    7.0   
785  St. Vincent and the Grenadines  2020    5.0   

                                    filename  
190                     Comoros_2021_October  
378                          Haiti_2022_July  
785  St. Vincent and the Grenadines_2020_May  


######  Export DF

In [57]:
# Cell A: Build df_export
meta_cols = ['filename', 'country', 'year', 'month', 'ifs', 'doc_type',
             'region', 'arrangement_number', 'review',
             'paired_with', 'paired_filename',
             'paired_conditional', 'paired_conditional_filename',
             'cex', 'fcs', 'sds', 'rst', 'hipc', 'frontier_market', 'exposure_china',
             'word_count']

available_meta = [c for c in meta_cols if c in df_mentions_all.columns]

topic_cols = []
for tk in MENTION_TOPICS:
    topic_cols.append(f'has_{tk}')
    topic_cols.append(f'count_{tk}')
    topic_cols.append(f'count_{tk}_per100w')

ordered_cols = available_meta + topic_cols
ordered_cols = [c for c in ordered_cols if c in df_mentions_all.columns]

df_export = df_mentions_all[ordered_cols]
print(f"Raw Data: {df_export.shape}")

Raw Data: (2601, 225)


In [58]:
# Cell B: Build df_agg
rows_agg = []

for doc_type in ['DSA', 'AIV', 'Program']:
    df_dt = df_export[df_export['doc_type'] == doc_type].copy()
    
    for tk in MENTION_TOPICS:
        col_raw = f'count_{tk}'
        col_norm = f'count_{tk}_per100w'
        
        cy = df_dt.groupby(['country', 'year']).agg(
            raw=(col_raw, 'mean'),
            norm=(col_norm, 'mean'),
        ).reset_index()
        
        yearly = cy.groupby('year').agg(
            median_raw=('raw', 'median'),
            mean_raw=('raw', 'mean'),
            p25_raw=('raw', lambda x: x.quantile(0.25)),
            p75_raw=('raw', lambda x: x.quantile(0.75)),
            median_norm=('norm', 'median'),
            mean_norm=('norm', 'mean'),
            p25_norm=('norm', lambda x: x.quantile(0.25)),
            p75_norm=('norm', lambda x: x.quantile(0.75)),
            n_countries=('raw', 'count'),
        ).reset_index()
        
        yearly['doc_type'] = doc_type
        yearly['topic'] = tk
        yearly['topic_label'] = MENTION_TOPICS[tk]['description']
        yearly['grouping'] = 'all_lics'
        yearly['group_value'] = 'All'
        rows_agg.append(yearly)
        
        for grouping in ['cex', 'fcs', 'sds', 'hipc', 'frontier_market', 'exposure_china']:
            for val in [1, 0]:
                sub = df_dt[df_dt[grouping] == val]
                if len(sub) == 0:
                    continue
                cy_g = sub.groupby(['country', 'year']).agg(
                    raw=(col_raw, 'mean'),
                    norm=(col_norm, 'mean'),
                ).reset_index()
                
                yearly_g = cy_g.groupby('year').agg(
                    median_raw=('raw', 'median'),
                    mean_raw=('raw', 'mean'),
                    p25_raw=('raw', lambda x: x.quantile(0.25)),
                    p75_raw=('raw', lambda x: x.quantile(0.75)),
                    median_norm=('norm', 'median'),
                    mean_norm=('norm', 'mean'),
                    p25_norm=('norm', lambda x: x.quantile(0.25)),
                    p75_norm=('norm', lambda x: x.quantile(0.75)),
                    n_countries=('raw', 'count'),
                ).reset_index()
                
                yearly_g['doc_type'] = doc_type
                yearly_g['topic'] = tk
                yearly_g['topic_label'] = MENTION_TOPICS[tk]['description']
                yearly_g['grouping'] = grouping
                yearly_g['group_value'] = str(val)
                rows_agg.append(yearly_g)

df_agg = pd.concat(rows_agg, ignore_index=True)
df_agg = df_agg[['doc_type', 'grouping', 'group_value', 'topic', 'topic_label', 'year',
                  'n_countries', 'median_raw', 'mean_raw', 'p25_raw', 'p75_raw',
                  'median_norm', 'mean_norm', 'p25_norm', 'p75_norm']]

print(f"Aggregated: {df_agg.shape}")

Aggregated: (54128, 15)


In [59]:
# Cell C: Save
with pd.ExcelWriter(OUTPUT_DIR / 'mentions_all_doctypes.xlsx') as writer:
    df_export.to_excel(writer, sheet_name='Raw Data', index=False)
    df_agg.to_excel(writer, sheet_name='Aggregated', index=False)
print(f"✓ Saved: {OUTPUT_DIR / 'mentions_all_doctypes.xlsx'}")

✓ Saved: U:\dil\bp2\output\mentions_all_doctypes.xlsx


In [ ]:
### 6 Pairing dataframe DSA -> AIV/Program

### 6 Comparacion del flujo de mensaje entre DSA y AIV/Program

In [61]:
# Build paired dataset
has_cols = [c for c in df_mentions_all.columns if c.startswith('has_')]
count_cols = [c for c in df_mentions_all.columns if c.startswith('count_')]

# Get parent filename (prefer paired_with, fallback to paired_conditional)
dsa_rows = df_mentions_all[df_mentions_all['doc_type'] == 'DSA'].copy()
dsa_rows['parent_filename'] = dsa_rows['paired_filename'].fillna(dsa_rows['paired_conditional_filename'])
dsa_rows['parent_type'] = dsa_rows['paired_with'].fillna(dsa_rows['paired_conditional'])

# Drop unpaired
dsa_rows = dsa_rows[dsa_rows['parent_filename'].notna()].copy()

# Parent lookup: filename -> row
parent_lookup = df_mentions_all.set_index('filename')[has_cols + count_cols]

# Build paired df
meta = dsa_rows[['filename', 'country', 'ifs', 'year', 'month', 
                  'parent_filename', 'parent_type',
                  'cex', 'fcs', 'sds', 'hipc', 'frontier_market', 'exposure_china']].copy()

# DSA columns
dsa_vals = dsa_rows[has_cols + count_cols].copy()
dsa_vals.columns = [f'{c}_dsa' for c in dsa_vals.columns]

# Parent columns
parent_vals = dsa_rows['parent_filename'].map(lambda x: parent_lookup.loc[x] if x in parent_lookup.index else None)
df_parent = pd.DataFrame(parent_vals.tolist(), index=dsa_rows.index)
df_parent.columns = [f'{c}_parent' for c in df_parent.columns]

# Combine
df_pairs = pd.concat([meta.reset_index(drop=True), 
                       dsa_vals.reset_index(drop=True), 
                       df_parent.reset_index(drop=True)], axis=1)

print(f"Paired dataset: {df_pairs.shape}")
print(f"  Paired with AIV: {(df_pairs['parent_type'] == 'AIV').sum()}")
print(f"  Paired with Program: {(df_pairs['parent_type'] == 'Program').sum()}")

Paired dataset: (921, 421)
  Paired with AIV: 389
  Paired with Program: 532


In [ ]:
# Quick view
#print(df_pairs[['filename', 'country', 'year', 'month', 'parent_filename', 'parent_type']].head(20))

# Export
df_pairs.to_excel(OUTPUT_DIR / 'dsa_parent_pairs.xlsx', index=False)
print(f"✓ Saved: {OUTPUT_DIR / 'dsa_parent_pairs.xlsx'}")

#### Topic aparece si o no 1/0

In [64]:
# Level 1: For each topic, when DSA mentions it, does the parent also mention it?
results = []

for tk in MENTION_TOPICS:
    has_dsa = f'has_{tk}_dsa'
    has_parent = f'has_{tk}_parent'
    
    # DSAs that mention this topic
    dsa_mentions = df_pairs[df_pairs[has_dsa] == 1]
    n_dsa = len(dsa_mentions)
    
    if n_dsa == 0:
        continue
    
    # Of those, how many parents also mention it
    n_parent = (dsa_mentions[has_parent] == 1).sum()
    
    results.append({
        'topic': tk,
        'topic_label': MENTION_TOPICS[tk]['description'],
        'n_dsa_mentions': n_dsa,
        'n_parent_mentions': n_parent,
        'transmission_rate': n_parent / n_dsa * 100,
    })

df_transmission = pd.DataFrame(results).sort_values('transmission_rate', ascending=False)
print(df_transmission[['topic_label', 'n_dsa_mentions', 'n_parent_mentions', 'transmission_rate']].to_string())

                             topic_label  n_dsa_mentions  n_parent_mentions  transmission_rate
63                   Trinidad and Tobago               8                  8         100.000000
59                               Morocco               2                  2         100.000000
25                Asian Development Bank             118                117          99.152542
23               World Bank / IDA / IBRD             882                874          99.092971
24              African Development Bank             282                277          98.226950
0               Global & Regional Shocks             616                605          98.214286
16        Debt Transparency & Disclosure             868                851          98.041475
4                          Domestic Debt             856                829          96.845794
35                             Australia              36                 34          94.444444
9           Political and Social Factors          

In [65]:
for parent_type in ['AIV', 'Program']:
    results = []
    sub = df_pairs[df_pairs['parent_type'] == parent_type]
    
    for tk in MENTION_TOPICS:
        has_dsa = f'has_{tk}_dsa'
        has_parent = f'has_{tk}_parent'
        
        dsa_mentions = sub[sub[has_dsa] == 1]
        n_dsa = len(dsa_mentions)
        if n_dsa == 0:
            continue
        
        n_parent = (dsa_mentions[has_parent] == 1).sum()
        results.append({
            'topic_label': MENTION_TOPICS[tk]['description'],
            'n_dsa': n_dsa,
            'transmission_rate': n_parent / n_dsa * 100,
        })
    
    df_t = pd.DataFrame(results).sort_values('transmission_rate', ascending=False)
    print(f"\n{'='*60}")
    print(f"TRANSMISSION: DSA → {parent_type}")
    print(f"{'='*60}")
    print(df_t[['topic_label', 'n_dsa', 'transmission_rate']].to_string())


TRANSMISSION: DSA → AIV
                             topic_label  n_dsa  transmission_rate
9           Political and Social Factors     60         100.000000
60                   Trinidad and Tobago      8         100.000000
58                              Portugal      3         100.000000
59                          South Africa     13         100.000000
55                             Argentina      1         100.000000
23               World Bank / IDA / IBRD    376          99.468085
25                Asian Development Bank    101          99.009901
0               Global & Regional Shocks    258          98.062016
35                             Australia     35          97.142857
16        Debt Transparency & Disclosure    371          96.765499
24              African Development Bank     70          94.285714
4                          Domestic Debt    357          92.997199
6                            Remittances    164          92.682927
2      Climate & Natural Disaster Sho

#### Topic Intensidad

In [66]:
# Level 2: Intensity comparison — count_per100w DSA vs Parent
results_intensity = []

for tk in MENTION_TOPICS:
    count_dsa = f'count_{tk}_per100w_dsa'
    count_parent = f'count_{tk}_per100w_parent'
    
    # Only pairs where DSA mentions the topic
    sub = df_pairs[df_pairs[f'has_{tk}_dsa'] == 1]
    n = len(sub)
    if n == 0:
        continue
    
    avg_dsa = sub[count_dsa].mean()
    avg_parent = sub[count_parent].mean()
    ratio = avg_parent / avg_dsa if avg_dsa > 0 else None
    
    results_intensity.append({
        'topic': tk,
        'topic_label': MENTION_TOPICS[tk]['description'],
        'n_pairs': n,
        'avg_per100w_dsa': round(avg_dsa, 4),
        'avg_per100w_parent': round(avg_parent, 4),
        'ratio': round(ratio, 2) if ratio else None,
    })

df_intensity = pd.DataFrame(results_intensity).sort_values('ratio', ascending=False)
print(df_intensity[['topic_label', 'n_pairs', 'avg_per100w_dsa', 'avg_per100w_parent', 'ratio']].to_string())

                             topic_label  n_pairs  avg_per100w_dsa  avg_per100w_parent  ratio
62                          South Africa       23           0.0830              0.1244   1.50
35                             Australia       36           0.0542              0.0740   1.37
11                               Arrears      856           0.1793              0.1915   1.07
16        Debt Transparency & Disclosure      868           0.0817              0.0869   1.06
1                          Health Shocks      243           0.2542              0.2527   0.99
53                                 India      108           0.0689              0.0674   0.98
9           Political and Social Factors      159           0.0435              0.0355   0.82
10                   Conflict & Violence       76           0.0377              0.0289   0.77
0               Global & Regional Shocks      616           0.0742              0.0523   0.71
2      Climate & Natural Disaster Shocks      244           